# 🌿 Hierarchical Clustering
**ISLP Chapter 12 · Data Pattern Recognition for the Rest of Us**

> Hierarchical clustering builds a tree of clusters — you can explore any number of segments by cutting the dendrogram at different heights. Unlike K-means, you don't need to specify K upfront.

### Dataset
**S&P 500 Stock Returns — Sector Clustering**
We cluster major US stocks by their daily return correlations. Stocks that move together get clustered together — revealing sector structure and diversification opportunities. This is how portfolio managers and risk analysts actually use clustering.

### Time: ~55 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
import scipy.stats as stats
print("\u2713 Setup complete")

In [ ]:
# S&P 500 representative stocks — synthetic returns with realistic sector correlations
# Stocks within the same sector have correlated returns; cross-sector correlation is lower
np.random.seed(42)
n_days = 252  # one trading year

stocks = {
    'Tech':       ['AAPL','MSFT','GOOGL','META','NVDA','AMZN'],
    'Finance':    ['JPM','BAC','GS','MS','WFC','C'],
    'Healthcare': ['JNJ','PFE','UNH','ABT','MRK','TMO'],
    'Energy':     ['XOM','CVX','COP','SLB','PSX','VLO'],
    'Consumer':   ['PG','KO','PEP','WMT','COST','MCD'],
}

all_tickers = [t for tickers in stocks.values() for t in tickers]
sector_labels = {t: s for s, tickers in stocks.items() for t in tickers}

# Market factor (all stocks move somewhat together)
market_factor = np.random.normal(0, 0.008, n_days)

# Generate correlated returns
returns_data = {}
for sector, tickers in stocks.items():
    # Sector factor — stocks in same sector move together
    sector_factor = np.random.normal(0, 0.006, n_days)
    for ticker in tickers:
        # Individual noise
        idio = np.random.normal(0, 0.012, n_days)
        # Energy has negative correlation with market during crises
        mkt_beta = -0.3 if sector == 'Energy' else 0.8
        returns_data[ticker] = mkt_beta*market_factor + 0.6*sector_factor + idio

returns = pd.DataFrame(returns_data, index=pd.date_range('2023-01-01', periods=n_days, freq='B'))

print("S&P 500 Representative Portfolio")
print(f"  {len(all_tickers)} stocks across {len(stocks)} sectors")
print(f"  {n_days} trading days of simulated returns")
print(f"\nSectors: {list(stocks.keys())}")
print(f"\nAverage daily return: {returns.mean().mean()*100:.3f}%")
print(f"Average daily volatility: {returns.std().mean()*100:.2f}%")

## 📐 Part 1 — Clustering Stocks by Return Correlation

Instead of clustering raw return values, we cluster by **correlation distance**:
```
distance(i,j) = 1 - correlation(i,j)
```
Stocks that move together (high correlation) have low distance and cluster together.
This is the standard approach in portfolio risk analysis.

In [ ]:
# Compute correlation matrix and convert to distance
corr_matrix = returns.corr()
dist_matrix = 1 - corr_matrix

# Hierarchical clustering — complete linkage
Z = linkage(squareform(dist_matrix.values), method='complete')

# Plot dendrogram
fig, ax = plt.subplots(figsize=(14, 5))
sector_colors = {'Tech':'#1e5fa8','Finance':'#e85d20','Healthcare':'#1a7a45',
                 'Energy':'#6b46c1','Consumer':'#0e7490'}
leaf_colors = [sector_colors[sector_labels[t]] for t in all_tickers]

dend = dendrogram(Z, labels=all_tickers, ax=ax, leaf_rotation=45,
                  color_threshold=0.7*max(Z[:,2]))
ax.set_title('Hierarchical Clustering of S&P 500 Stocks by Return Correlation\n'
             'Height = dissimilarity — stocks that merge early are most similar')
ax.set_ylabel('Correlation Distance (1 - correlation)')

# Add sector color legend
for sector, color in sector_colors.items():
    ax.plot([], [], 's', color=color, label=sector, markersize=8)
ax.legend(title='Actual Sector', loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()
print("\n\u2714 Stocks within the same sector cluster together at low height")
print("   Energy separates early (negative market correlation)")
print("   This confirms the sector structure without using any sector labels")

In [ ]:
# Cut dendrogram at different heights = different numbers of clusters
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, n_clusters in zip(axes, [3, 5, 7]):
    cluster_labels = fcluster(Z, n_clusters, criterion='maxclust')

    # Show correlation heatmap colored by cluster
    idx_sorted = np.argsort(cluster_labels)
    sorted_corr = corr_matrix.iloc[idx_sorted, idx_sorted]
    sorted_tickers = [all_tickers[i] for i in idx_sorted]

    im = ax.imshow(sorted_corr.values, cmap='RdYlGn', vmin=-0.5, vmax=1.0, aspect='auto')
    ax.set_xticks(range(len(sorted_tickers)))
    ax.set_yticks(range(len(sorted_tickers)))
    ax.set_xticklabels(sorted_tickers, rotation=90, fontsize=7)
    ax.set_yticklabels(sorted_tickers, fontsize=7)
    ax.set_title(f'{n_clusters} Clusters\n(sorted by cluster membership)')

    # Add cluster boundaries
    boundaries = np.where(np.diff(cluster_labels[idx_sorted]))[0] + 0.5
    for b in boundaries:
        ax.axhline(b, color='black', lw=1.5)
        ax.axvline(b, color='black', lw=1.5)

plt.colorbar(im, ax=axes[-1], label='Correlation', shrink=0.8)
plt.suptitle('Correlation Heatmaps at Different Cluster Cuts\n'
             'Green = high correlation, Red = negative correlation', fontsize=11, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Portfolio diversification insight
n_clusters = 5
cluster_labels = fcluster(Z, n_clusters, criterion='maxclust')
stock_clusters = pd.DataFrame({'Ticker': all_tickers,
                                'Sector': [sector_labels[t] for t in all_tickers],
                                'Cluster': cluster_labels})

print("=== Cluster Composition ===\n")
for cl in sorted(stock_clusters['Cluster'].unique()):
    members = stock_clusters[stock_clusters['Cluster']==cl]
    print(f"Cluster {cl} ({len(members)} stocks):")
    for _, row in members.iterrows():
        print(f"  {row['Ticker']:<8} [{row['Sector']}]")
    # Within-cluster avg correlation
    tickers = members['Ticker'].tolist()
    if len(tickers) > 1:
        avg_corr = corr_matrix.loc[tickers, tickers].values
        np.fill_diagonal(avg_corr, np.nan)
        print(f"  Avg within-cluster correlation: {np.nanmean(avg_corr):.3f}")
    print()

print("\u2714 Portfolio insight: pick ONE stock per cluster for maximum diversification")
print("   Stocks in the same cluster move together — owning multiple adds little diversification benefit")

In [ ]:
# @title 📝 Quiz — Hierarchical Clustering
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is the key advantage of hierarchical over K-means clustering?
# @markdown **Q2:** What does complete linkage measure when merging two clusters?
# @markdown **Q3:** Why do we cluster by correlation rather than raw return levels?
# @markdown **Q4:** How does a dendrogram help choose the number of clusters?
# @markdown **Q5:** If two stocks are in the same cluster, what does that mean for portfolio diversification?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "Hierarchical Clustering"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*